In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from dateutil.parser import parse
import importlib
import app.prg_ibbi

importlib.reload(app.prg_ibbi)



from app.prg_ibbi import IBBI
from app.utils import Helper

utils = Helper()
config_path =r"..\cestat\config\config_ibbi.json5" 
if __name__ == "__main__":

    config = utils.load_json5(config_path) 
    ibbi = IBBI(config)
    results = ibbi.get_data()

    # ---- SAVE OUTPUT ----
    output_file = "ibbi_orders.xlsx"
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        for name, df in results.items():
            df.to_excel(writer, sheet_name=name[:31], index=False)
            print(f"✔ {name}: {len(df)} rows")

    print(f"\n Done. Data saved to {output_file}")


In [ ]:
import os
import importlib
from app.jobs.job_ibbi import IBBIJob
import app.prg_ibbi

importlib.reload(app.prg_ibbi)
importlib.reload(app.jobs.job_ibbi)

from app.utils import Helper
from app.logger import setup_logger
from app.mailer import Mailer
from app.konstant import DATA_DIR


def ibbi_runner():

    utils = Helper()
    config_path = r"..\rep_cestat\config\config_ibbi.json5"
    config = utils.load_json5(config_path)

    print(config)
    
    logger = setup_logger(name="ibbi", set_global=True)


    mailer = Mailer({"send_mail":False})

    # --- job ---
    job = IBBIJob(
        data_dir=os.path.join(DATA_DIR, "ibbi"),
        config=config,
        logger=logger,
        mailer=mailer
    )

    # --- run ---
    return job.run()


if __name__ == "__main__":
    ibbi_runner()

COMPARE THE DATASET

In [3]:
import pandas as pd
import hashlib
import os

file_old = r"C:\Users\rando\Downloads\IBBI_ALL_20260421.xlsx"
file_new = r"C:\Users\rando\Downloads\IBBI_ALL_20260422.xlsx"
output_dir = "compare_output"

os.makedirs(output_dir, exist_ok=True)

# -------------------------
# HASH FUNCTION
# -------------------------
def generate_hash(row):
    values = []

    for v in row[2:-2]:  # ignore section, srno, hash_id, is_new
        if pd.isna(v):
            v = ""
        v = str(v).strip().lower()
        values.append(v)

    return hashlib.md5("|".join(values).encode()).hexdigest()


# -------------------------
# LOAD SINGLE SHEET
# -------------------------
old_df = pd.read_excel(file_old, sheet_name="ibbi")
new_df = pd.read_excel(file_new, sheet_name="ibbi")

# -------------------------
# GENERATE HASHES (SAFE)
# -------------------------
old_df = old_df.copy()
new_df = new_df.copy()

old_df["hash_id"] = [generate_hash(r) for r in old_df.values]
new_df["hash_id"] = [generate_hash(r) for r in new_df.values]

# -------------------------
# COMPARE
# -------------------------
old_hashes = set(old_df["hash_id"])
new_hashes = set(new_df["hash_id"])

new_df["status"] = new_df["hash_id"].apply(
    lambda x: "NEW" if x not in old_hashes else "COMMON"
)

removed_df = old_df[~old_df["hash_id"].isin(new_hashes)].copy()
removed_df["status"] = "REMOVED"

# -------------------------
# FINAL OUTPUT
# -------------------------
final_df = pd.concat([new_df, removed_df], ignore_index=True)

final_df = final_df.sort_values(
    by="status",
    key=lambda col: col.map({"NEW": 0, "REMOVED": 1, "COMMON": 2})
)

out_path = os.path.join(output_dir, "ibbi_compare.csv")
final_df.to_csv(out_path, index=False)

print(f"[DONE] → {out_path}")

[DONE] → compare_output\ibbi_compare.csv


In [9]:
"https://www.nseindia.com/api/corporates/offerdocs?index=debt&from_date=20-10-2025&to_date=20-04-2026"

'https://www.nseindia.com/api/corporates/offerdocs?index=debt&from_date=20-10-2025&to_date=20-04-2026'